# Notebook 3 — three frequency groups, no redistribution

Level-1 code: $B$, $V$, $IR$ packets through one slab with $\kappa_B > \kappa_V > \kappa_{IR}$. First pure extinction, $I_g = I_{g,0} e^{-\tau_g}$; then elastic scattering with the frequency held fixed. A packet never changes its group here — that happens in chapter 5.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[0] / "src"))   # rtedu, uninstalled (education/src)
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import rtedu
from rtedu import results
from rtedu.visualization import save_fig, OI, GROUP_COLOUR
rng = np.random.default_rng(rtedu.SEEDS["ch03"])

In [ ]:
depth = 1.0
kappa = dict(B=3.0, V=1.0, IR=0.3)           # per unit length; the ordering is the physics
wavelength_nm = dict(B=440.0, V=550.0, IR=1250.0)
n_per_group = 20_000
tau = {g: kappa[g] * depth for g in kappa}
transmitted = {}
for g in kappa:                                # pure extinction: one draw per packet
    tau_int = -np.log(1.0 - rng.random(n_per_group))
    transmitted[g] = float(np.mean(tau_int > tau[g]))
for g in kappa:
    print(f"{g:>2s}: tau = {tau[g]:.2f}  transmitted {transmitted[g]:.4f}  e^-tau {np.exp(-tau[g]):.4f}")

## Elastic scattering, frequency held fixed

Now every interaction re-emits the packet isotropically at the *same* frequency. More opacity means more interactions and a longer path before escape, but the emergent packet is still a $B$ (or $V$, or $IR$) packet.

In [ ]:
def scatter_slab(rng, tau_slab, n):
    x = np.zeros(n); mu = np.ones(n); alive = np.ones(n, bool)
    n_int = np.zeros(n, int); path = np.zeros(n); fate = np.zeros(n, int)
    while alive.any():
        idx = np.flatnonzero(alive)
        s = -np.log(1.0 - rng.random(idx.size))
        x_new = x[idx] + mu[idx] * s; path[idx] += s
        out = x_new >= tau_slab; back = x_new < 0.0
        path[idx[out]] -= (x_new[out] - tau_slab) / mu[idx[out]]; path[idx[back]] -= x_new[back] / mu[idx[back]]
        fate[idx[out]] = 1; fate[idx[back]] = 2; alive[idx[out | back]] = False
        stay = idx[~(out | back)]; x[stay] = x_new[~(out | back)]; n_int[stay] += 1
        mu[stay] = rng.uniform(-1.0, 1.0, stay.size)        # elastic: new direction, same frequency
    tr = fate == 1
    return dict(transmitted=float(tr.mean()), reflected=float((fate == 2).mean()),
                mean_interactions=float(n_int[tr].mean()), mean_path=float(path[tr].mean()), paths=path[tr])

scat = {g: scatter_slab(rng, tau[g], n_per_group) for g in kappa}
for g in kappa:
    print(f"{g:>2s}: transmitted {scat[g]['transmitted']:.3f}  reflected {scat[g]['reflected']:.3f}  interactions {scat[g]['mean_interactions']:.2f}  path {scat[g]['mean_path']:.2f} mfp")

## Validation against `rtedu`

`ThreeGroupSlab.run` is the same bookkeeping; with `albedo=0` it is pure extinction, with `albedo=1` pure elastic scattering.

In [ ]:
from rtedu.slab import ThreeGroupSlab
tg = ThreeGroupSlab(depth, kappa["B"], kappa["V"], kappa["IR"])
ext = tg.run(rng, n_per_group, albedo=0.0); sc = tg.run(rng, n_per_group, albedo=1.0)
for g in kappa:
    p = np.exp(-tau[g]); sig = np.sqrt(p * (1 - p) / n_per_group)
    assert abs(transmitted[g] - p) < 4 * sig and abs(ext[g]["transmitted"] - p) < 4 * sig
assert sc["B"]["mean_interactions_transmitted"] > sc["V"]["mean_interactions_transmitted"] > sc["IR"]["mean_interactions_transmitted"]
print({g: round(sc[g]["mean_interactions_transmitted"], 2) for g in kappa})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.5))
gs = list(kappa); cols = [GROUP_COLOUR[g] for g in gs]
axes[0].bar([wavelength_nm[g] for g in gs], [kappa[g] for g in gs], width=80, color=cols); axes[0].set_xlabel("wavelength [nm]"); axes[0].set_ylabel(r"$\kappa$"); axes[0].set_title("opacity per group", fontsize=9)
w = 0.35; xg = np.arange(3)
axes[1].bar(xg - w / 2, [1.0] * 3, w, color="lightgrey", label="injected"); axes[1].bar(xg + w / 2, [transmitted[g] for g in gs], w, color=cols, label="transmitted")
axes[1].set_xticks(xg); axes[1].set_xticklabels(gs); axes[1].set_ylabel("fraction"); axes[1].set_title("input vs output SED, pure extinction", fontsize=9); axes[1].legend()
for g in gs:
    axes[2].hist(scat[g]["paths"], bins=np.linspace(0, 12, 49), histtype="step", color=GROUP_COLOUR[g], label=f"{g}: mean {scat[g]['mean_path']:.1f}")
axes[2].set_xlabel("path length of transmitted packets [mean free paths]"); axes[2].set_ylabel("packets"); axes[2].set_title("elastic scattering: escape path by group", fontsize=9); axes[2].legend(fontsize=8)
fig.tight_layout(); save_fig(fig, "ch03_three_groups")

In [ ]:
results.record("ch03", dict(depth=depth, kappa=kappa, wavelength_nm=wavelength_nm, n_per_group=n_per_group, tau=tau,
                            transmitted=transmitted, transmission_exact={g: float(np.exp(-tau[g])) for g in kappa},
                            scattering={g: dict(transmitted=scat[g]["transmitted"], reflected=scat[g]["reflected"],
                                                mean_interactions=scat[g]["mean_interactions"], mean_path=scat[g]["mean_path"]) for g in kappa}))